In [10]:
import os
os.environ['HF_HOME'] = 'D:\\HuggingFace'
os.environ['TRANSFORMERS_CACHE'] = os.environ['HF_HOME']
os.environ['HUGGINGFACE_HUB_CACHE'] = os.environ['HF_HOME'] 

In [1]:
from warnings import filterwarnings
filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import pipeline
import re

sns.set_style('darkgrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

2026-03-08 18:55:27.033711: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772996127.222908      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772996127.277317      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [19]:
device

'cuda'

In [2]:
!pip install PyPDF2 # обработка пдф
!pip install chonkie # чанки

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.0/389.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 92.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
mkl-umath 0.1.1 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.4.2 which is incompatible.
mkl-random 1.2.4 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.4.2 which is incompatible.
mkl-fft 1.3.8 requires numpy<1.27.0,>=1.26.4, but you 

# Обработка pdf

In [11]:
clean_data = pd.DataFrame({"Algo": [], "Analysis": [], "NN": [], "Optim": [], "SQL": []})

In [3]:
from PyPDF2 import PdfReader


def process_info(file):
    with open(file, "rb") as f:
        reader = PdfReader(f)
        text = ''.join([page.extract_text() for page in reader.pages])
    # Служебная информация
    text = re.sub(r'ISSN\s+\d{4}-\d{3,4}[^\n]*', '', text)
    text = re.sub(r'\d{4};\d{2}\(\d+\):\d+–\d+', '', text)

    # Авторы
    text = re.sub(r'[А-ЯЁ][а-яё]+\s+[А-ЯЁ][\.\s]+\s*[А-ЯЁ][\.\s]*', '', text)
    text = re.sub(r'[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ][\.\s]+\s*[А-ЯЁ][\.\s]*', '', text)
    text = re.sub(r'\d+[\s\w\.,–-]+(университет|институт|академия|центр)[^\n]*', '', text)

    # Сноски в квадратных скобках
    text = re.sub(r'\[\d+\]', '', text)  # [1], [2]
    text = re.sub(r'\[\d+[,-]\d+\]', '', text)  # [1-3], [4,5]
    text = re.sub(r'\[[A-Za-z]+\d*\]', '', text)  # [A1], [B]
    
    # email
    text = re.sub(r'\S+@\S+', '', text)
    
    # английские разделы 
    text = re.sub(r'Abstract[^\n]*[\s\S]*?(?=\n[А-ЯЁ]|$)', '', text)
    text = re.sub(r'Keywords[^\n]*[\s\S]*?(?=\n[А-ЯЁ]|$)', '', text)
    text = re.sub(r'For citation[^\n]*[\s\S]*?(?=\n[А-ЯЁ]|$)', '', text)
    
    # ссылки
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'DOI:\s*\S+', '', text)
    
    # библиография
    text = re.sub(r'Список\s+источников[\s\S]*', '', text)
    text = re.sub(r'References[\s\S]*', '', text)
    
    # спец.символы
    text = text.replace('\xa0', ' ').replace('•', '')
    text = re.sub(r'-\s+', '', text)  
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'•\s*\n', '', text)
    
    # оставшиеся английские фрагменты
    text = re.sub(r'(?:[A-Za-z-]+\s){3,}[A-Za-z-]*', '', text)
    
    return text

In [5]:
from pathlib import Path


def make_clean_pdfs(path):
    books = Path(path)
    files_to_process = []
    for book in books.iterdir():
        files_to_process.append(str(book))

    clean_files = []
    for file in files_to_process:
        clean_files.append(process_info(file))
        
    return clean_files

## Обработка всех классов и очистка pdf

In [36]:
algo_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Алгоритмы и структуры данных"
analysis_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Анализ данных"
nn_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/ИИ"
optim_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Методы оптимизации"
sql_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/SQL"

In [37]:
clean_data['Algo'] = make_clean_pdfs(algo_path)
print("____________DONE_____________")
clean_data['Analysis'] = make_clean_pdfs(analysis_path)
print("____________DONE_____________")
clean_data['NN'] = make_clean_pdfs(nn_path)
print("____________DONE_____________")
clean_data['Optim'] = make_clean_pdfs(optim_path)
print("____________DONE_____________")
clean_data['SQL'] = make_clean_pdfs(sql_path)
print("____________DONE_____________")

FileNotFoundError: [Errno 2] No such file or directory: 'D:/ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Алгоритмы и структуры данных'

In [11]:
clean_data.to_csv('clean_pdfs.csv', sep=',', index=False, encoding='utf-8-sig', escapechar='\\')

In [9]:
clean_data.shape

(5, 5)

# Создание обучающей выборки из эмбеддингов документов

In [4]:
text_dataset = pd.read_csv('/kaggle/input/5-classes-diplom/clean_pdfs.csv')

In [5]:
print(text_dataset.shape)
print(text_dataset.columns)

(5, 5)
Index(['Algo', 'Analysis', 'NN', 'Optim', 'SQL'], dtype='object')


In [6]:
len(text_dataset['Algo'][4])

208985

## Чанкование

In [7]:
from chonkie import TokenChunker, OverlapRefinery

In [8]:
len(text_dataset['Algo'][0])

560770

In [7]:
# Инициализация базового чанкировщика
chunker = TokenChunker(
    chunk_size=512,
    chunk_overlap=50
)

# Разделение текста на чанки по базовым правилам TokenChunker
chunks = chunker(text_dataset['Algo'][0])

NameError: name 'text_dataset' is not defined

In [10]:
chunks[0]

Chunk(text='Алгоритмы и структуры данных Новая версия для Оберона + CD Москва, 2010Никлаус Вирт Перевод с английского под редакцией доктора физ-мат. наук, УДК 32.973.26-018.2 ББК 004.438 В52 Никлаус Вирт В52 Алгоритмы и структуры данных. Новая версия для Оберона + CD / Пер. с англ. – М.: ДМК П ресс, 2010. – 272 с.: ил. ISBN 978-5-94074-584-6 В классическом учебнике тьюринговского лауреата Н.Вирта аккуратно, на тщательно подобранных примерах прорабатываются основные темы алго)ритмики – сортировка и поиск, рекурсия, дина', token_count=512, start_index=0, end_index=512)

In [11]:
len(chunks)

1214

## Загрузка модели

In [4]:
from transformers import AutoModel, AutoTokenizer 

model_name = "DeepPavlov/rubert-base-cased-conversational" 

model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

inputs = tokenizer("Hello world!", return_tensors="pt")

outputs = model(**inputs)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased-conversational were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Анализ получения эмбеддингов

In [8]:
chunker = TokenChunker(
    chunk_size=512,
    chunk_overlap=50
)

In [14]:
embeddings = torch.mean(outputs.last_hidden_state, dim=1)
embedding_array = embeddings.detach().numpy()

In [15]:
embedding_array.shape

(1, 768)

In [16]:
inputs = tokenizer(str(chunks[0]), return_tensors="pt")
outputs = model(**inputs)
embeddings = torch.mean(outputs.last_hidden_state, dim=1)
embedding_array = embeddings.detach().numpy()
embedding_array.shape

(1, 768)

Проставим такие метки для классов:
- Algo - 0
- Analysis - 1
- NN - 2
- Optim - 3
- SQL - 4

In [17]:
classes_dict = {'Algo': 0, 'Analysis': 1, 'NN': 2, 'Optim': 3, 'SQL': 4}

In [5]:
def make_embeddings_from_document(document, label, device=None): 
    if not hasattr(make_embeddings_from_document, 'model_on_device'):
        model.to(device)
        make_embeddings_from_document.model_on_device = True
        
    embeddings = []
    labels = []
    chunks = chunker(document)
    print(f'Общая длина: {len(chunks)}')
    
    for i in range(len(chunks)):
        inputs = tokenizer(str(chunks[i]), return_tensors="pt").to(device)
        
        with torch.no_grad(): 
            outputs = model(**inputs)
            embeddings_from_outputs = torch.mean(outputs.last_hidden_state, dim=1)
            
            embedding_array = embeddings_from_outputs.cpu().numpy()

        # Засовываем в общий массив
        embeddings.append(embedding_array)
        labels.append(classes_dict[label])
        
        if i % 100 == 0:
            print(f'{i} есть👌')
            
    return embeddings, labels

In [19]:
embeddings, labels = make_embeddings_from_document(text_dataset['Algo'][0], 'Algo', device)

Общая длина: 1214
0 есть👌
100 есть👌
200 есть👌
300 есть👌


KeyboardInterrupt: 

## Создаём массивы со всеми эмбеддингами и метками

In [102]:
embeddings_list = []
labels_list = []
for class_name in classes_dict:
    print(f"Обрабатываем класс: {class_name}")
    texts = text_dataset[class_name] 

    for text in texts:
        embeddings, labels = make_embeddings_from_document(text, class_name, device)
        embeddings_list.extend(embeddings)
        labels_list.extend(labels)
    
    print(f'{class_name} отработал 👌')

Обрабатываем класс: Algo
Общая длина: 1214
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
800 есть👌
900 есть👌
1000 есть👌
1100 есть👌
1200 есть👌
Общая длина: 295
0 есть👌
100 есть👌
200 есть👌
Общая длина: 1337
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
800 есть👌
900 есть👌
1000 есть👌
1100 есть👌
1200 есть👌
1300 есть👌
Общая длина: 440
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
Общая длина: 453
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
Algo отработал 👌
Обрабатываем класс: Analysis
Общая длина: 540
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
Общая длина: 584
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
Общая длина: 423
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
Общая длина: 723
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
Общая длина: 2292
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
800 есть👌
900 есть👌
1000 есть👌
1100 ест

In [108]:
len(labels_list)

21254

# Сохранение в файл h5

In [109]:
import h5py
import numpy as np

with h5py.File('embeddings_dataset.h5', 'w') as f:
    f.create_dataset('embeddings', data=np.array(embeddings_list))
    f.create_dataset('labels', data=np.array(labels_list))

# Сохраняем rubert, токенизатор и словарь

In [5]:
model.save_pretrained('rubert_embedding')
tokenizer.save_pretrained('rubert_embedding')

('rubert_embedding/tokenizer_config.json',
 'rubert_embedding/special_tokens_map.json',
 'rubert_embedding/vocab.txt',
 'rubert_embedding/added_tokens.json',
 'rubert_embedding/tokenizer.json')

In [8]:
import pickle

with open('class_mapping.pkl', 'wb') as f:
    pickle.dump(classes_dict, f)

In [9]:
import shutil
shutil.make_archive('rubert_embedding', 'zip')

'rubert_embedding.zip'

# Сохраняем все данные

In [9]:
text_dataset = pd.read_csv('/kaggle/input/datasets/arturosaf8/full-classes-clean/full_clean_pdfs.csv')

In [10]:
from transformers import AutoModel, AutoTokenizer 

model_name = "DeepPavlov/rubert-base-cased-conversational" 

model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

inputs = tokenizer("Hello world!", return_tensors="pt")

outputs = model(**inputs)

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased-conversational were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [11]:
chunker = TokenChunker(
    chunk_size=512,
    chunk_overlap=50
)

In [13]:
text_dataset.columns

Index(['Algo', 'Analysis', 'NN', 'Optim', 'SQL', 'Software_engineering',
       'CIS_design', 'Software_testing', 'Project_management', 'Philosophy',
       'Python'],
      dtype='object')

Проставим такие метки для классов:
- Algo - 0
- Analysis - 1
- NN - 2
- Optim - 3
- SQL - 4
- Software_engineering - 5
- CIS_design - 6
- Software_testing - 7
- Project_management - 8
- Philosophy - 9
- Python - 10

In [12]:
classes_dict = {'Algo': 0, 'Analysis': 1, 'NN': 2, 'Optim': 3, 'SQL': 4,
                'Software_engineering': 5, 'CIS_design': 6, 'Software_testing': 7,
                'Project_management': 8, 'Philosophy': 9, 'Python': 10}

In [13]:
embeddings_list = []
labels_list = []
for class_name in classes_dict:
    print(f"Обрабатываем класс: {class_name}")
    texts = text_dataset[class_name] 

    for text in texts:
        embeddings, labels = make_embeddings_from_document(text, class_name, device)
        embeddings_list.extend(embeddings)
        labels_list.extend(labels)
    
    print(f'{class_name} отработал 👌')

Обрабатываем класс: Algo
Общая длина: 1214
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
800 есть👌
900 есть👌
1000 есть👌
1100 есть👌
1200 есть👌
Общая длина: 295
0 есть👌
100 есть👌
200 есть👌
Общая длина: 1337
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
800 есть👌
900 есть👌
1000 есть👌
1100 есть👌
1200 есть👌
1300 есть👌
Общая длина: 440
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
Общая длина: 453
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
Algo отработал 👌
Обрабатываем класс: Analysis
Общая длина: 540
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
Общая длина: 584
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
Общая длина: 423
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
Общая длина: 722
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
Общая длина: 2292
0 есть👌
100 есть👌
200 есть👌
300 есть👌
400 есть👌
500 есть👌
600 есть👌
700 есть👌
800 есть👌
900 есть👌
1000 есть👌
1100 ест

In [14]:
import h5py
import numpy as np

with h5py.File('full_embeddings_dataset.h5', 'w') as f:
    f.create_dataset('embeddings', data=np.array(embeddings_list))
    f.create_dataset('labels', data=np.array(labels_list))

In [18]:
import pickle

model.save_pretrained('rubert_embedding')
tokenizer.save_pretrained('rubert_embedding')

with open('class_mapping.pkl', 'wb') as f:
    pickle.dump(classes_dict, f)

In [16]:
%cd /kaggle/working
!zip embeddings.zip full_embeddings_dataset.h5

/kaggle/working
  adding: full_embeddings_dataset.h5

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)


In [17]:
from IPython.display import FileLink
FileLink('embeddings.zip')

/kaggle/working/embeddings.zip

In [21]:
%cd /kaggle/working
!zip -r rubert_embedding.zip rubert_embedding/

/kaggle/working
updating: rubert_embedding/ (stored 0%)
  adding: rubert_embedding/tokenizer_config.json (deflated 74%)
  adding: rubert_embedding/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 73%)
  adding: rubert_embedding/model.safetensors (deflated 8%)
  adding: rubert_embedding/vocab.txt (deflated 65%)
  adding: rubert_embedding/special_tokens_map.json (deflated 42%)
  adding: rubert_embedding/config.json (deflated 54%)


In [22]:
from IPython.display import FileLink
FileLink('rubert_embedding.zip')

/kaggle/working/rubert_embedding.zip